In [1]:
from pathlib import Path
import os
import random
import json
import pickle
from collections import Counter

import numpy as np
import pandas as pd

from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import timm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt
import seaborn as sns

/home/jamyoung/miniconda3/envs/yolov3/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ROOT = Path("../preprocess/preprocessing")
META_PATH = ROOT / "metadata.csv"

EFF_CLS_DIR = Path("outputs_effnetv2_m_finetune_classifier")
EFF_CLS_DIR.mkdir(parents=True, exist_ok=True)

classes = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
label2idx = {c: i for i, c in enumerate(classes)}
idx2label = {i: c for c, i in label2idx.items()}

seed = 42
model_name = "tf_efficientnetv2_m"

image_size = 300
batch_size = 8
num_workers = 0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(seed)

print("device:", device)
print("model_name:", model_name)
print("root:", ROOT)
print("metadata:", META_PATH)

device: cuda
model_name: tf_efficientnetv2_m
root: ../preprocess/preprocessing
metadata: ../preprocess/preprocessing/metadata.csv


In [3]:
df = pd.read_csv(META_PATH)

print(df.head())
print(df.columns)
print(df.shape)
print(df["dx"].value_counts())

required_cols = [
    "lesion_id",
    "image_id",
    "dx",
    "dx_type",
    "age",
    "sex",
    "localization",
    "dataset"
]

for col in required_cols:
    assert col in df.columns, f"Missing column: {col}"

def get_image_path(row):
    cls = row["dx"]
    image_id = row["image_id"]
    path = ROOT / cls / "enhanced" / f"{image_id}.jpg"
    if path.exists():
        return str(path)
    return None

df["image_path"] = df.apply(get_image_path, axis=1)

missing = df["image_path"].isna().sum()
print("missing images:", missing)

if missing > 0:
    df[df["image_path"].isna()][["image_id", "dx", "image_path"]].to_csv(
        EFF_CLS_DIR / "missing_images.csv",
        index=False
    )
    raise FileNotFoundError(f"{missing} images missing. Check missing_images.csv")

df["label"] = df["dx"].map(label2idx).astype(int)

print(df[["image_id", "dx", "label", "image_path"]].head())
print(df["label"].value_counts().sort_index())

     lesion_id      image_id   dx dx_type   age   sex localization  \
0  HAM_0000118  ISIC_0027419  bkl   histo  80.0  male        scalp   
1  HAM_0000118  ISIC_0025030  bkl   histo  80.0  male        scalp   
2  HAM_0002730  ISIC_0026769  bkl   histo  80.0  male        scalp   
3  HAM_0002730  ISIC_0025661  bkl   histo  80.0  male        scalp   
4  HAM_0001466  ISIC_0031633  bkl   histo  75.0  male          ear   

        dataset  
0  vidir_modern  
1  vidir_modern  
2  vidir_modern  
3  vidir_modern  
4  vidir_modern  
Index(['lesion_id', 'image_id', 'dx', 'dx_type', 'age', 'sex', 'localization',
       'dataset'],
      dtype='object')
(10015, 8)
dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64
missing images: 0
       image_id   dx  label                                         image_path
0  ISIC_0027419  bkl      2  ../preprocess/preprocessing/bkl/enhanced/ISIC_...
1  ISIC_0025030  bkl      2  ../prepr

In [4]:
train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    stratify=df["dx"],
    random_state=seed
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("train:", len(train_df))
print(train_df["dx"].value_counts())

print("\nval:", len(val_df))
print(val_df["dx"].value_counts())

train_df.to_csv(EFF_CLS_DIR / "train_split.csv", index=False)
val_df.to_csv(EFF_CLS_DIR / "val_split.csv", index=False)

train: 9013
dx
nv       6034
mel      1002
bkl       989
bcc       463
akiec     294
vasc      128
df        103
Name: count, dtype: int64

val: 1002
dx
nv       671
mel      111
bkl      110
bcc       51
akiec     33
vasc      14
df        12
Name: count, dtype: int64


In [5]:
train_tfms = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(30),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10,
        hue=0.03
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_tfms = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [6]:
class HAMImageDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["image_path"]).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(int(row["label"]), dtype=torch.long)
        image_id = row["image_id"]

        return image, label, image_id

In [7]:
train_dataset_cls = HAMImageDataset(train_df, transform=train_tfms)
val_dataset_cls = HAMImageDataset(val_df, transform=val_tfms)

train_loader_cls = DataLoader(
    train_dataset_cls,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True
)

val_loader_cls = DataLoader(
    val_dataset_cls,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True
)

batch = next(iter(train_loader_cls))
images, labels, image_ids = batch

print(images.shape)
print(labels.shape)
print(image_ids[:3])
print(labels[:10])

torch.Size([8, 3, 300, 300])
torch.Size([8])
['ISIC_0028345', 'ISIC_0031165', 'ISIC_0028873']
tensor([5, 5, 5, 1, 5, 4, 5, 5])


In [8]:
class_counts = train_df["label"].value_counts().sort_index().values
N = class_counts.sum()
C = len(classes)

class_weights = N / (C * class_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float32)

for cls, count, weight in zip(classes, class_counts, class_weights):
    print(f"{cls:6s} count={count:5d}, weight={weight.item():.4f}")

akiec  count=  294, weight=4.3795
bcc    count=  463, weight=2.7809
bkl    count=  989, weight=1.3019
df     count=  103, weight=12.5007
mel    count= 1002, weight=1.2850
nv     count= 6034, weight=0.2134
vasc   count=  128, weight=10.0592
